In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "DIQ_C.xpt"
csv_file = DATA_DIR / "DIQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched copy for fidelity comparison
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 DIQ_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED RELEASED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "DIQ010",
    "DID040G",
    "DID040Q",
    "DIQ050",
    "DID060G",
    "DID060Q",
    "DIQ060U",
    "DIQ070",
    "DIQ080",
    "DIQ090",
    "DIQ100",
    "DIQ110",
    "DIQ120",
    "DIQ130",
    "DIQ140",
    "DIQ150"
]


if df.columns.tolist() == EXPECTED_COLUMNS:
    print("\nPASS: Column names/order match DIQ_C.")
else:
    raise ValueError(
        "DIQ_C column names/order do not match expected structure."
    )


# ============================================================
# 4. KNOWN PANDAS/XPT ZERO ARTIFACT CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


total_tiny = int(
    tiny_counts.sum()
)

total_zero = int(
    zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{total_zero:,}"
)

print(
    "Tiny artifact values:",
    f"{total_tiny:,}"
)


if total_tiny > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    # DIQ_C attached source does not contain legitimate
    # pandas tiny-zero artifacts.
    #
    # Do not silently change anything.

    raise ValueError(
        "STOP: Unexpected tiny-value artifact found in DIQ_C. "
        "Do NOT automatically replace it with zero."
    )


print(
    "PASS: No tiny zero-decoding artifact exists."
)


# ============================================================
# 5. VERIFY NO GENUINE DECIMAL VALUES
# ============================================================

#
# Important fidelity rule:
#
# Do not convert a field to integer merely because
# it looks like a questionnaire variable.
#
# First verify all actual source values are whole numbers.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values)
    )

    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nUnexpected decimals in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values found. "
        "These columns were NOT converted:\n"
        + str(fractional_columns)
    )


print(
    "PASS: All DIQ_C source values are "
    "whole-number codes/quantities."
)


# ============================================================
# 6. CLEAN SEQN
# ============================================================

if "SEQN" not in df.columns:
    raise ValueError(
        "SEQN not found."
    )


seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):
    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 7. FORMAT QUESTIONNAIRE / INTEGER QUANTITY FIELDS
# ============================================================

#
# All released DIQ fields in THIS XPT were independently
# checked above and contain only whole-number values.
#
# Examples:
#
# 1.0  -> 1
# 47.0 -> 47
# NaN  -> blank CSV field
#
# This does NOT round a genuine decimal because the script
# already stops if one exists.
#

DIQ_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in DIQ_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


print(
    "\nPASS: DIQ questionnaire codes/quantities "
    "formatted without unnecessary .0."
)


# ============================================================
# 8. CDC STRUCTURE CHECK
# ============================================================

print("\n--- CDC STRUCTURE CHECK ---")


if len(df) == 9645:
    print(
        "PASS: Row count = 9,645"
    )
else:
    print(
        "FAIL: Row count =",
        len(df)
    )


if len(df.columns) == 17:
    print(
        "PASS: Released column count = 17"
    )
else:
    print(
        "FAIL: Column count =",
        len(df.columns)
    )


# ============================================================
# 9. CDC FREQUENCY DEFINITIONS
# ============================================================

EXPECTED = {

    "DIQ010": {
        "counts": {
            1: 559,
            2: 9001,
            3: 78,
            9: 7
        },
        "missing": 0
    },

    "DID040G": {
        "counts": {
            1: 548,
            2: 8,
            9: 3
        },
        "missing": 9086
    },

    "DIQ050": {
        "counts": {
            1: 144,
            2: 9501
        },
        "missing": 0
    },

    "DID060G": {
        "counts": {
            1: 143,
            9: 1
        },
        "missing": 9501
    },

    "DIQ060U": {
        "counts": {
            1: 14,
            2: 129
        },
        "missing": 9502
    },

    "DIQ070": {
        "counts": {
            1: 395,
            2: 163,
            9: 1
        },
        "missing": 9086
    },

    "DIQ080": {
        "counts": {
            1: 120,
            2: 421,
            9: 4
        },
        "missing": 9100
    },

    "DIQ090": {
        "counts": {
            1: 141,
            2: 3154,
            9: 4
        },
        "missing": 6346
    },

    "DIQ100": {
        "counts": {
            1: 775,
            2: 2521,
            9: 3
        },
        "missing": 6346
    },

    "DIQ110": {
        "counts": {
            1: 350,
            2: 163,
            3: 262
        },
        "missing": 8870
    },

    "DIQ120": {
        "counts": {
            1: 770,
            2: 2525,
            9: 4
        },
        "missing": 6346
    },

    "DIQ130": {
        "counts": {
            1: 355,
            2: 175,
            3: 240
        },
        "missing": 8875
    },

    "DIQ140": {
        "counts": {
            1: 990,
            2: 2302,
            9: 7
        },
        "missing": 6346
    },

    "DIQ150": {
        "counts": {
            1: 585,
            2: 404,
            9: 1
        },
        "missing": 8655
    }
}


# ============================================================
# 10. RUN CDC FREQUENCY CHECKS
# ============================================================

print("\n--- CDC FREQUENCY CHECK ---")


frequency_failures = []


for col, spec in EXPECTED.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    counts_match = (
        actual_counts
        ==
        spec["counts"]
    )

    missing_match = (
        actual_missing
        ==
        spec["missing"]
    )


    if counts_match and missing_match:

        print(
            f"PASS: {col}"
        )

    else:

        frequency_failures.append(
            col
        )

        print(
            f"\nFAIL: {col}"
        )

        print(
            "Expected counts:",
            spec["counts"]
        )

        print(
            "Actual counts:",
            actual_counts
        )

        print(
            "Expected missing:",
            spec["missing"]
        )

        print(
            "Actual missing:",
            actual_missing
        )


if not frequency_failures:

    print(
        "\nPASS: All categorical DIQ variables "
        "match CDC frequencies."
    )


# ============================================================
# 11. DID040Q RANGE / MISSING CHECK
# ============================================================

#
# CDC:
#
# 1-88  = 548 records
# Missing = 9,097
#

s = df["DID040Q"]


print("\n--- DID040Q ---")

print(
    "Valid:",
    int(s.notna().sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)


if (
    int(s.notna().sum()) == 548
    and
    int(s.isna().sum()) == 9097
    and
    s.min() == 1
    and
    s.max() == 88
):

    print(
        "PASS: DID040Q matches CDC."
    )

else:

    print(
        "FAIL: DID040Q differs from CDC."
    )


# ============================================================
# 12. DID060Q RANGE / MISSING CHECK
# ============================================================

#
# CDC:
#
# 1-47 = 143 records
# Missing = 9,502
#

s = df["DID060Q"]


print("\n--- DID060Q ---")

print(
    "Valid:",
    int(s.notna().sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)


if (
    int(s.notna().sum()) == 143
    and
    int(s.isna().sum()) == 9502
    and
    s.min() == 1
    and
    s.max() == 47
):

    print(
        "PASS: DID060Q matches CDC."
    )

else:

    print(
        "FAIL: DID060Q differs from CDC."
    )


# ============================================================
# 13. VERIFY ORIGINAL XPT VALUES WERE NOT CHANGED
# ============================================================

#
# There is NO zero restoration in DIQ_C.
#
# Therefore expected values are exactly the original XPT
# numeric values.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


for col in source_df.columns:

    source_values = (
        source_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        source_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} value changed."
        )


# ============================================================
# 14. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL TINY-VALUE CHECK ---")

print(
    "Remaining tiny values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny-value artifact exists."
    )

else:

    raise ValueError(
        "FAIL: Tiny values remain."
    )


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# No float_format.
# No rounding of genuine decimal measurements.
#
# DIQ has no genuine decimal fields in this XPT.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 16. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 17. EXACT XPT -> CSV VALUE FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


value_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches XPT."
        )

    else:

        value_failures.append(
            col
        )

        print(
            f"FAIL: {col} differs from XPT."
        )


if not value_failures:

    print(
        "\nPASS: Every DIQ_C numeric value "
        "matches the source XPT."
    )


# ============================================================
# 18. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )

else:

    raise ValueError(
        "FAIL: Missing-value positions changed."
    )


# ============================================================
# 19. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count == 0:

    print(
        "PASS: No unnecessary .0 formatting."
    )

else:

    print(
        "CHECK: .0 formatting remains."
    )


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific-notation occurrences:",
    scientific_count
)


if scientific_count == 0:

    print(
        "PASS: No scientific notation."
    )


print("\n" + "=" * 75)
print("DIQ_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 DIQ_C XPT -> CSV
Rows: 9,645
Columns: 17

Variables:
['SEQN', 'DIQ010', 'DID040G', 'DID040Q', 'DIQ050', 'DID060G', 'DID060Q', 'DIQ060U', 'DIQ070', 'DIQ080', 'DIQ090', 'DIQ100', 'DIQ110', 'DIQ120', 'DIQ130', 'DIQ140', 'DIQ150']

PASS: Column names/order match DIQ_C.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact values: 0
PASS: No tiny zero-decoding artifact exists.

--- FRACTIONAL VALUE CHECK ---
PASS: All DIQ_C source values are whole-number codes/quantities.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31126

PASS: DIQ questionnaire codes/quantities formatted without unnecessary .0.

--- CDC STRUCTURE CHECK ---
PASS: Row count = 9,645
PASS: Released column count = 17

--- CDC FREQUENCY CHECK ---
PASS: DIQ010
PASS: DID040G
PASS: DIQ050
PASS: DID060G
PASS: DIQ060U
PASS: DIQ070
PASS: DIQ080
PASS: DIQ090
PASS: DIQ100
PASS: DIQ110
PASS: DIQ120
PASS: DIQ130
PASS: DIQ140
PASS: DIQ150

PASS: All categorical DIQ variables match CDC freq